# MoE 라우팅과 부하균형 실습 - 추가 실습 코드: MoE Routing 시뮬레이션

- Tutorial ID: `mod-moe-routing-lab`
- Tutorial: MoE 라우팅과 부하균형 실습
- Section ID: `practice-moe-routing`
- Section: 추가 실습 코드: MoE Routing 시뮬레이션


In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: MoE Routing 시뮬레이션
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   2) router가 토큰을 expert에 배정하는 과정과 load balancing 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def top_k_routing(hidden_states, router_weights, k=2, num_experts=8):
    """
    MoE Top-K Routing
    
    Args:
        hidden_states: (batch_size, hidden_dim)
        router_weights: (hidden_dim, num_experts)
        k: 각 토큰당 활성화할 expert 수
    """
        batch_size = hidden_states.shape[0]
    
    # Router logits 계산
        router_logits = np.matmul(hidden_states, router_weights)
    
    # Top-K expert 선택
    top_k_indices = np.argsort(router_logits, axis=-1)[:, -k:]
    
    # Gating weights (Top-K만 유지, 나머지는 0)
        gating_weights = np.zeros_like(router_logits)
        for i in range(batch_size):
        gating_weights[i, top_k_indices[i]] = router_logits[i, top_k_indices[i]]
    
    # Softmax (Top-K 내에서 정규화)
        gating_weights = softmax(gating_weights)
    
    return top_k_indices, gating_weights, router_logits

# 시뮬레이션
np.random.seed(42)
batch_size = 4
hidden_dim = 64
num_experts = 8
k = 2

# 임의의 hidden states와 router weights
hidden_states = np.random.randn(batch_size, hidden_dim)
router_weights = np.random.randn(hidden_dim, num_experts) * 0.1

print("=" * 60)
print(f"MoE Top-{k} Routing (Experts: {num_experts})")
print("=" * 60)

indices, weights, logits = top_k_routing(
    hidden_states, router_weights, k=k, num_experts=num_experts
)

print(f"\\n각 토큰이 선택한 Expert와 가중치:\\n")
for i in range(batch_size):
    experts = indices[i]
        probs = weights[i, experts]
    bar1 = "█" * int(probs[0] * 20)
    bar2 = "█" * int(probs[1] * 20)
    print(f"  Token {i}: Expert {experts[0]} ({probs[0]:.2f}) {bar1}")
    print(f"           Expert {experts[1]} ({probs[1]:.2f}) {bar2}")
    print()

# Load Balancing 분석
expert_loads = np.zeros(num_experts)
for i in range(batch_size):
        for j in indices[i]:
        expert_loads[j] += 1

print("Expert 부하 분포:")
for e in range(num_experts):
    bar = "█" * int(expert_loads[e] * 4)
    print(f"  Expert {e}: {int(expert_loads[e])} 토큰 {bar}")

print("\\n💡 MoE 핵심:")
print(f"  - 전체 {num_experts} Expert 중 {k}개만 활성화")
print(f"  - 연산량 ~1/{num_experts//k} 감소!")
print(f"  - Load Balancing Loss로 균등 분배 유도")